# Isolation Forest - Detecção de Anomalias

Após clustering, os dados serão processados para detectar anomalias isoladas, ajustando contamination (0,05–0,1) com base na análise exploratória. 

O algoritmo identificará picos de contaminação (ex.: fósforo elevado), focando em eventos críticos. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pcj.utils import BASE_DIR, dados

In [ ]:
# Dados para testes

dados_limpos, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")

X = dados_limpos.copy()

In [ ]:
# Verificações

# print(dados_limpos['Data'].dtype, end='')
# print(dados_limpos['data_normalizada'].dtype)
# print(dados_limpos.columns)
# print(variaveis)
print(dados_limpos[variaveis])

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 01 - Minha Tentativa

In [ ]:
from sklearn.ensemble import IsolationForest

X = dados_limpos[variaveis].copy()

isf = IsolationForest(contamination=0.05).fit(X)

isf.predict(X)

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 02 - Seguindo Tutorial

Link para o tutorial:
https://medium.com/datarisk-io/detec%C3%A7%C3%A3o-de-anomalias-usando-o-isolation-forest-5d29728fb255

Na verdade, o tutorial não deixa muito claro como executar as etapas que ele lista, vou tomar uma liberdade poética.

In [ ]:
# Estabelecendo as regras do algoritmo (acho)
from sklearn.ensemble import IsolationForest

iFo = IsolationForest(contamination=0.05,   # No projeto, queremos testar esse valor de 0,05 a 0,1
                      random_state=42       # É o valor usado no tutorial, e percebo que ele é frequentemente usado. Imagino que seja um padrão por conta do Hitchiker's Guide to the Galaxy       
                      )

In [ ]:
# Rótulos binários (mais parecido com o tutorial)

rotulos = iFo.fit_predict(dados_limpos[variaveis])   # Faz o fit e classificação, gerando rótulos binários (-1 e 1)

outliers = dados_limpos[rotulos == -1]

outliers

In [ ]:
# Scores contínuos

scores = iFo.fit(dados_limpos[variaveis]) # Faz o fit, retorna valores contínuos para os scores das anomalias. Permite escolher os limites para anomalias.

scores.score_samples(dados_limpos[variaveis])

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 03 - Loops para os métodos acima

Talvez haja um pouco de redundância

In [ ]:
# Setup

from sklearn.ensemble import IsolationForest
X = dados_limpos[variaveis].copy()

valores_contaminacao = [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]


In [ ]:
# 01 - Minha Tentativa:

resultados_1 = {}

for i in valores_contaminacao:
    isf = IsolationForest(contamination=i).fit(X)
    rotulos = isf.predict(X)

    n_outliers = (rotulos == -1).sum()
    pct_outliers = (n_outliers / len(X)) * 100

    resultados_1[i] = {
        'modelo': isf,
        'rótulos': rotulos,
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers
    }

    print(f"Contamination {i}: {n_outliers} outliers, {pct_outliers:.1f}%)")

# Seleção do melhor contamination

best_contam = max(resultados_1, key=lambda x: resultados_1[x]['n_outliers'])    # Definir a lógica correta para identificar qual é o melhor valor
                                                                            # Nesse caso, ele seleciona o contamination que gera o maior número de outliers.
print(f"\n Contamination ideal: {best_contam}")

In [ ]:
# 02 - Rótulos binários (mais parecido com o tutorial):

resultados_2 = {}

for i in valores_contaminacao:
    isf = IsolationForest(contamination=i).fit(X)

    rotulos = isf.fit_predict(X)   # Faz o fit e classificação, gerando rótulos binários (-1 e 1)

    n_outliers = (rotulos == -1).sum()

    pct_outliers = (n_outliers / len(X)) * 100

    resultados_2[i] = {
        'modelo': isf,
        'rótulos': rotulos,
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers
    }

    print(f"Contamination {i}: {n_outliers} outliers, {pct_outliers:.1f}%)")

# Seleção do melhor contamination

best_contam = max(resultados_2, key=lambda x: resultados_2[x]['n_outliers'])    # Definir a lógica correta para identificar qual é o melhor valor
                                                                            # Nesse caso, ele seleciona o contamination que gera o maior número de outliers.
print(f"\n Contamination ideal: {best_contam}")

In [ ]:
# 03 - Scores contínuos

resultados_3 = {}

for i in valores_contaminacao:
    isf = IsolationForest(contamination=i).fit(X)

    scores = isf.fit(X) # Faz o fit, retorna valores contínuos para os scores das anomalias. Permite escolher os limites para anomalias.

    scores_samples = scores.score_samples(X)
    # n_outliers = (scores_samples < -0.5).sum() # Selecionar o threshold adequado para determinar outliers. -0.5 é um placeholder
    threshold = np.percentile(scores_samples, 100 * i)  # Esse método é menos arbitrário, e calcula um threshold em função do contamination
    outlier_mask = scores_samples < threshold
    n_outliers = np.sum(outlier_mask)

    pct_outliers = (n_outliers / len(X)) * 100

    resultados_3[i] = {
        'modelo': isf,
        # 'scores': scores, # Não sei se encaixa aqui, estou tentando reciclar o código de antes, mas não sei se isso cabe
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers
    }

    print(f"Contamination {i}: {n_outliers} outliers, {pct_outliers:.1f}%)")

# Seleção do melhor contamination

best_contam = max(resultados_3, key=lambda x: resultados_3[x]['n_outliers'])    # Definir a lógica correta para identificar qual é o melhor valor
                                                                            # Nesse caso, ele seleciona o contamination que gera o maior número de outliers.
print(f"\n Contamination ideal: {best_contam}")

### 04 - Seguindo outro tutorial, com Visualizações

Link para o tutorial: https://ai.plainenglish.io/unsupervised-outlier-detection-with-isolation-forest-eab398c593b2

In [ ]:
# Estabelecimento dos parâmetros do modelo

from sklearn.ensemble import IsolationForest

X = dados_limpos[variaveis].copy()

iforest = IsolationForest(n_estimators=100,     # Copiando do tutorial
                          contamination=0.05,   # Dentro do que queremos no projeto
                          max_samples='auto'    # Copiando do tutorial
                          )

In [ ]:
# Rodando o modelo

prediction = iforest.fit_predict(X)

print("Número de outliers detectados: {}".format(prediction[prediction < 0].sum()))

print("Número de amostras normais detectadass: {}".format(prediction[prediction > 0].sum()))

In [ ]:
# Plotando os resultados

# amostras_normais = X[np.where(prediction > 0)]    # Não estão funcionando. O tutorial usa um pd.Series, então a indexação
# outliers = X[np.where(prediction < 0)]            # quebra em pd.DataFrame

amostras_normais = X.iloc[np.where(prediction > 0)[0]]  # iloc faz a indexação funcionar no pd.DataFrame 
outliers = X.iloc[np.where(prediction < 0)[0]]

plt.scatter(amostras_normais.iloc[:, 0],  # Seleciona a primeira coluna do dataframe
            amostras_normais.iloc[:, 1],  # Seleciona a segunda coluna
            s=10,   # Define um tamanho para os pontos normais. Será menor do que o dos outliers para facilitar visualização
            label='normal', 
            alpha=0.6  # Define opacidade dos pontos
            )  
plt.scatter(outliers.iloc[:, 0], outliers.iloc[:, 1], s=20, color='r', label='outlier', alpha=0.9)
plt.legend()
plt.title(f'IsolationForest: {variaveis[0]} vs {variaveis[1]}')
plt.show()

In [ ]:
# Como há diversas variáveis, esses scatterplots de 2 dimensões seriam limitados.
# Entretanto, podemos usar PCA para reduzir todas as variáveis para 2 dimensões.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X)

X_scaled = scaler.transform(X)

pca = PCA(n_components=2).fit_transform(X_scaled)
pred = iforest.fit_predict(X)

plt.scatter(pca[pred > 0, 0], pca[pred > 0, 1], s=10, label='normal')
plt.scatter(pca[pred < 0, 0], pca[pred < 0, 1], s=20, color='r', label='outlier')
plt.legend()
plt.title('PCA (2D) + IsolationForest')
plt.show()

In [ ]:
# Isolation Forest com série temporal. Gráficos individuais para cada variável

pred = iforest.fit_predict(X)

normal_idx = X.index[pred > 0]
outlier_idx = X.index[pred < 0]

for var in variaveis:
    plt.figure(figsize=(12, 3))
    plt.scatter(dados_limpos.loc[normal_idx, 'data_normalizada'],
                X.loc[normal_idx, var], s=10, label='normal')
    plt.scatter(dados_limpos.loc[outlier_idx, 'data_normalizada'],
                X.loc[outlier_idx, var], s=30, color='r', label='outlier')
    plt.title(var)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()